Celda 1: Generación y Modelado de Ítems (raw_synthetic_items_prep)
Esta vista temporal crea las posiciones con sus métricas calculadas y las anomalías inyectadas.

(Nota: Viene configurado por defecto en 10.000 órdenes (~25.000 posiciones) para resguardar la memoria del cluster gratuito. Para generar las 100.000 solo debes cambiar sequence(1, 10000) por sequence(1, 100000)).

A partir de ahora se trabaja en la version final

A. Probar la demo para reconocer errores.
B. Una vez que se vea funcional se tienen que agregar las columnas que aparezcan en silver, tanto tmp como stg, + modificar algunos parametros si lo requiere y la evalucion de contenido de columnas para modelar de cierta forma en funcion de los tipos de datos que tienen que ir en sus respectivas columans
ej: si estoy con algun amount se que seran numeros que se transforman a decimal, al igual que con los pesos weight.
C. Una vez funcional hacer la medicion de consumos para sumar a proyecciones y decidir como actuar.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW raw_synthetic_items_prep AS
WITH orders_seed AS (
    SELECT 
        id AS order_id,
        concat('90', lpad(cast(id AS STRING), 8, '0')) AS SalesOrder,
        pmod(abs(hash(id, 'day_seed')), 578) AS day_offset,
        concat('900', lpad(cast(1 + pmod(abs(hash(id, 'cust_seed')), 50) AS STRING), 3, '0')) AS SoldToParty,
        CASE WHEN pmod(abs(hash(id, 'dist_seed')), 10) < 7 THEN '90' ELSE '91' END AS DistributionChannel,
        1 + pmod(abs(hash(id, 'items_cnt')), 4) AS num_items,
        pmod(abs(hash(id, 'anomaly_seed')), 200) AS anomaly_code
    FROM (SELECT explode(sequence(1, 100000)) AS id)
),
items_expanded AS (
    SELECT 
        o.order_id,
        o.SalesOrder,
        o.day_offset,
        o.SoldToParty,
        o.DistributionChannel,
        o.anomaly_code,
        item_idx,
        
        -- Anomalía 1 (Radio 2 - Hard): SalesOrderItem NULL
        CASE 
            WHEN o.anomaly_code = 0 AND item_idx = 1 THEN NULL 
            ELSE cast(item_idx * 10 AS STRING) 
        END AS SalesOrderItem,
        
        concat('900', lpad(cast(1 + pmod(abs(hash(o.order_id, item_idx, 'mat')), 40) AS STRING), 2, '0')) AS Material,
        
        -- Anomalía 2 (Radio 4 - Hard): Cantidad negativa
        CASE 
            WHEN o.anomaly_code = 1 AND item_idx = 1 THEN -5.0
            ELSE cast(1 + pmod(abs(hash(o.order_id, item_idx, 'qty')), 15) AS DECIMAL(18, 3))
        END AS RequestedQuantity,
        
        cast(40.0 + (pmod(abs(hash(o.order_id, item_idx, 'price')), 51000) / 100.0) AS DECIMAL(18, 2)) AS UnitPrice,
        cast(0.20 + (pmod(abs(hash(o.order_id, item_idx, 'margin')), 11) / 100.0) AS DECIMAL(18, 4)) AS MarginPct,
        
        CASE 
            WHEN pmod(abs(hash(o.order_id, item_idx, 'deliv_st')), 10) < 7 THEN 'C'
            WHEN pmod(abs(hash(o.order_id, item_idx, 'deliv_st')), 10) < 9 THEN 'B'
            ELSE 'A'
        END AS DeliveryStatus,
        
        CASE WHEN pmod(abs(hash(o.order_id, item_idx, 'proc_st')), 4) = 0 THEN 'B' ELSE 'C' END AS SDProcessStatus,
        CASE WHEN pmod(abs(hash(o.order_id, item_idx, 'bill_st')), 3) = 0 THEN 'A' ELSE 'C' END AS BillingStatus
    FROM orders_seed o
    LATERAL VIEW explode(sequence(1, o.num_items)) t AS item_idx
)
SELECT 
    order_id,
    SalesOrder,
    SalesOrderItem,
    Material,
    day_offset,
    SoldToParty,
    DistributionChannel,
    anomaly_code,
    'TAN' AS SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    BillingStatus,
    
    -- Disparador SCD Tipo 2 en Producto (Material 90025 cambia en día 300)
    CASE 
        WHEN Material = '90025' AND day_offset >= 300 THEN 'L009'
        ELSE concat('L00', cast(1 + pmod(abs(hash(Material, 'mgroup')), 3) AS STRING))
    END AS MaterialGroup,
    
    CASE WHEN pmod(abs(hash(Material, 'prgrp')), 2) = 0 THEN 'A1' ELSE 'B2' END AS MaterialPricingGroup,
    Material AS OriginallyRequestedMaterial,
    
    -- MaterialByCustomer poblado con códigos de cliente estándar
    concat('CUST-MAT-', Material) AS MaterialByCustomer,
    
    cast(RequestedQuantity AS STRING) AS RequestedQuantity,
    cast(RequestedQuantity AS STRING) AS ConfdDelivQtyInOrderQtyUnit,
    cast(round(RequestedQuantity * UnitPrice, 2) AS STRING) AS NetAmount,
    cast(round((RequestedQuantity * UnitPrice) * 0.21, 2) AS STRING) AS TaxAmount,
    cast(round((RequestedQuantity * UnitPrice) * (1.0 - MarginPct), 2) AS STRING) AS CostAmount,
    cast(round(abs(RequestedQuantity) * 1.550, 3) AS STRING) AS ItemGrossWeight,
    cast(round(abs(RequestedQuantity) * 1.400, 3) AS STRING) AS ItemNetWeight,
    cast(round(abs(RequestedQuantity) * 0.035, 3) AS STRING) AS ItemVolume,
    
    -- Anomalía 3 (Radio 6 - Hard): Corrupción técnica
    CASE 
        WHEN anomaly_code = 2 AND SalesOrderItem = '10' THEN '{"error": "CORRUPTED_SYNTHETIC_PAYLOAD"}'
        ELSE NULL 
    END AS _rescued_data
FROM items_expanded;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW raw_synthetic_headers_prep AS
WITH header_aggregation AS (
    SELECT 
        order_id,
        SalesOrder,
        day_offset,
        SoldToParty,
        DistributionChannel,
        max(anomaly_code) AS anomaly_code,
        sum(cast(NetAmount AS DECIMAL(18, 2))) AS CalculatedTotalNet,
        max(_rescued_data) AS HeaderRescuedData
    FROM raw_synthetic_items_prep
    GROUP BY order_id, SalesOrder, day_offset, SoldToParty, DistributionChannel
),
dates_and_metadata AS (
    SELECT 
        SalesOrder,
        '9000' AS SalesOrganization,
        DistributionChannel,
        '09' AS OrganizationDivision,
        SoldToParty,
        
        -- CustomerPaymentTerms distribuido entre 0001, 0002, 0003 y 0004
        -- Preservando el disparador SCD Tipo 2 para el cliente 900015
        CASE 
            WHEN SoldToParty = '900015' AND day_offset >= 250 THEN '0001'
            WHEN SoldToParty = '900015' THEN '0004'
            ELSE concat('000', cast(1 + pmod(abs(hash(SoldToParty, 'terms')), 4) AS STRING))
        END AS CustomerPaymentTerms,
        
        -- CustomerGroup diversificado (01 a 04)
        concat('0', cast(1 + pmod(abs(hash(SoldToParty, 'grp')), 4) AS STRING)) AS CustomerGroup,
        
        -- CustomerAccountAssignmentGroup según canal (01 Doméstico, 02 Exterior)
        CASE WHEN DistributionChannel = '91' THEN '02' ELSE '01' END AS CustomerAccountAssignmentGroup,

        -- Simular un 2% de pedidos cancelados/rechazados (Soft Delete)
        CASE 
            WHEN pmod(abs(hash(order_id, 'rejection_seed')), 50) = 0 THEN 'C'
            ELSE 'A' 
        END AS OverallSDDocumentRejectionSts,
        
        (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) + (pmod(abs(hash(order_id, 'sec')), 36000) * 1000)) AS CreationEpochMs,
        
        -- Anomalía 4 (Radio 6 - Hard): Entrega en el pasado
        CASE 
            WHEN anomaly_code = 3 THEN 
                (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) - (10 * 86400000))
            ELSE 
                (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) + (cast(3 + pmod(abs(hash(order_id, 'deliv_days')), 12) AS BIGINT) * 86400000))
        END AS DeliveryEpochMs,
        
        -- Anomalía 5 (Radio 5 - Soft): Descuadre contable
        CASE 
            WHEN anomaly_code = 4 THEN CalculatedTotalNet + 250.00
            ELSE CalculatedTotalNet 
        END AS TotalNetAmount,
        
        -- Anomalía 6 (Radio 3 - Hard): Moneda inválida
        CASE 
            WHEN anomaly_code = 5 THEN 'US_DOLLAR'
            ELSE 'USD'
        END AS TransactionCurrency,
        
        HeaderRescuedData AS _rescued_data
    FROM header_aggregation
)
SELECT 
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    concat('/Date(', cast(CreationEpochMs AS STRING), ')/') AS CreationDate,
    concat('/Date(', cast(CreationEpochMs AS STRING), ')/') AS SalesOrderDate,
    concat('/Date(', cast(DeliveryEpochMs AS STRING), ')/') AS RequestedDeliveryDate,
    cast(round(TotalNetAmount, 2) AS STRING) AS TotalNetAmount,
    TransactionCurrency,
    _rescued_data
FROM dates_and_metadata;

In [0]:
%sql
INSERT INTO workspace.bronze.sales_order_header (
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    CreationDate,
    SalesOrderDate,
    RequestedDeliveryDate,
    TotalNetAmount,
    TransactionCurrency,
    OverallSDDocumentRejectionSts,
    _rescued_data,
    _ingestion_timestamp,
    SourceSystem,
    IsSynthetic
)
SELECT 
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    CreationDate,
    SalesOrderDate,
    RequestedDeliveryDate,
    TotalNetAmount,
    TransactionCurrency,
    OverallSDDocumentRejectionSts,
    _rescued_data,
    current_timestamp() AS _ingestion_timestamp,
    'SYNTHETIC_GENERATOR' AS SourceSystem,
    TRUE AS IsSynthetic
FROM raw_synthetic_headers_prep;

In [0]:
%sql
INSERT INTO workspace.bronze.sales_order_item (
    SalesOrder,
    SalesOrderItem,
    Material,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    MaterialByCustomer,
    SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    OrderRelatedBillingStatus,
    RequestedQuantity,
    ConfdDelivQtyInOrderQtyUnit,
    NetAmount,
    TaxAmount,
    CostAmount,
    ItemGrossWeight,
    ItemNetWeight,
    ItemVolume,
    _rescued_data,
    _ingestion_timestamp,
    SourceSystem,
    IsSynthetic
)
-- 1. Posiciones sintéticas regulares
SELECT 
    SalesOrder,
    SalesOrderItem,
    Material,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    MaterialByCustomer,
    SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    BillingStatus,
    RequestedQuantity,
    ConfdDelivQtyInOrderQtyUnit,
    NetAmount,
    TaxAmount,
    CostAmount,
    ItemGrossWeight,
    ItemNetWeight,
    ItemVolume,
    _rescued_data,
    current_timestamp() AS _ingestion_timestamp,
    'SYNTHETIC_GENERATOR' AS SourceSystem,
    TRUE AS IsSynthetic
FROM raw_synthetic_items_prep

UNION ALL

-- 2. Inyección intencional de 3 registros duplicados con timestamp más reciente
SELECT 
    SalesOrder,
    SalesOrderItem,
    Material,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    MaterialByCustomer,
    SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    BillingStatus,
    RequestedQuantity,
    ConfdDelivQtyInOrderQtyUnit,
    NetAmount,
    TaxAmount,
    CostAmount,
    ItemGrossWeight,
    ItemNetWeight,
    ItemVolume,
    _rescued_data,
    current_timestamp() + INTERVAL 5 SECOND AS _ingestion_timestamp,
    'SYNTHETIC_GENERATOR' AS SourceSystem,
    TRUE AS IsSynthetic
FROM raw_synthetic_items_prep
WHERE SalesOrder IN ('9000000010', '9000000020', '9000000030')
  AND SalesOrderItem = '10';

In [0]:
%sql
-- 1. Balance de registros en Bronze por Sistema de Origen
SELECT 
    SourceSystem,
    IsSynthetic,
    COUNT(DISTINCT SalesOrder) AS TotalOrdenes,
    min(SalesOrder) AS MinOrden,
    max(SalesOrder) AS MaxOrden
FROM workspace.bronze.sales_order_header
GROUP BY SourceSystem, IsSynthetic;

In [0]:
%sql
-- 2. Inspección de anomalías inyectadas listas para Cuarentena
SELECT 
    h.SalesOrder,
    h.TransactionCurrency,
    h.TotalNetAmount AS HeaderAmount,
    round(sum(cast(i.NetAmount AS DECIMAL(18,2))), 2) AS SumItemAmount,
    count(i.SalesOrderItem) AS TotalItems,
    max(CASE WHEN i.RequestedQuantity LIKE '-%' THEN 'CANTIDAD_NEGATIVA' END) AS AlertaMetrica,
    max(CASE WHEN i._rescued_data IS NOT NULL THEN 'RESCUED_DATA' END) AS AlertaCorrupcion
FROM workspace.bronze.sales_order_header h
LEFT JOIN workspace.bronze.sales_order_item i ON h.SalesOrder = i.SalesOrder
WHERE h.IsSynthetic = TRUE
GROUP BY h.SalesOrder, h.TransactionCurrency, h.TotalNetAmount
HAVING 
    h.TransactionCurrency != 'USD'
    OR abs(cast(h.TotalNetAmount AS DECIMAL(18,2)) - sum(cast(i.NetAmount AS DECIMAL(18,2)))) > 0.01
    OR AlertaMetrica IS NOT NULL
    OR AlertaCorrupcion IS NOT NULL
LIMIT 10;

In [0]:
%sql
select * from workspace.bronze.sales_order_header where CreationDate > '/Date(1470009599)/'

anexion parte 2 a partir de 01/08/2016

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW raw_synthetic_items_prep AS
WITH orders_seed AS (
    SELECT 
        id AS order_id,
        -- Numeración consecutiva: 9000100001 en adelante
        concat('90', lpad(cast(id AS STRING), 8, '0')) AS SalesOrder,
        -- Desplazamiento: Día 578 (01/08/2016) al Día 1460 (31/12/2018)
        578 + pmod(abs(hash(id, 'day_seed')), 883) AS day_offset,
        -- Cartera de 50 clientes sintéticos
        concat('900', lpad(cast(1 + pmod(abs(hash(id, 'cust_seed')), 50) AS STRING), 3, '0')) AS SoldToParty,
        -- Canales de distribución
        CASE WHEN pmod(abs(hash(id, 'dist_seed')), 10) < 7 THEN '90' ELSE '91' END AS DistributionChannel,
        -- 1 a 4 posiciones por pedido
        1 + pmod(abs(hash(id, 'items_cnt')), 4) AS num_items,
        pmod(abs(hash(id, 'anomaly_seed')), 200) AS anomaly_code
    FROM (SELECT explode(sequence(100001, 250000)) AS id)
),
items_expanded AS (
    SELECT 
        o.order_id,
        o.SalesOrder,
        o.day_offset,
        o.SoldToParty,
        o.DistributionChannel,
        o.anomaly_code,
        item_idx,
        
        -- Anomalía 1 (Radio 2 - Hard): SalesOrderItem NULL
        CASE 
            WHEN o.anomaly_code = 0 AND item_idx = 1 THEN NULL 
            ELSE cast(item_idx * 10 AS STRING) 
        END AS SalesOrderItem,
        
        concat('900', lpad(cast(1 + pmod(abs(hash(o.order_id, item_idx, 'mat')), 40) AS STRING), 2, '0')) AS Material,
        
        -- Anomalía 2 (Radio 4 - Hard): Cantidad negativa
        CASE 
            WHEN o.anomaly_code = 1 AND item_idx = 1 THEN -5.0
            ELSE cast(1 + pmod(abs(hash(o.order_id, item_idx, 'qty')), 15) AS DECIMAL(18, 3))
        END AS RequestedQuantity,
        
        cast(40.0 + (pmod(abs(hash(o.order_id, item_idx, 'price')), 51000) / 100.0) AS DECIMAL(18, 2)) AS UnitPrice,
        cast(0.20 + (pmod(abs(hash(o.order_id, item_idx, 'margin')), 11) / 100.0) AS DECIMAL(18, 4)) AS MarginPct,
        
        CASE 
            WHEN pmod(abs(hash(o.order_id, item_idx, 'deliv_st')), 10) < 7 THEN 'C'
            WHEN pmod(abs(hash(o.order_id, item_idx, 'deliv_st')), 10) < 9 THEN 'B'
            ELSE 'A'
        END AS DeliveryStatus,
        
        CASE WHEN pmod(abs(hash(o.order_id, item_idx, 'proc_st')), 4) = 0 THEN 'B' ELSE 'C' END AS SDProcessStatus,
        CASE WHEN pmod(abs(hash(o.order_id, item_idx, 'bill_st')), 3) = 0 THEN 'A' ELSE 'C' END AS BillingStatus
    FROM orders_seed o
    LATERAL VIEW explode(sequence(1, o.num_items)) t AS item_idx
)
SELECT 
    order_id,
    SalesOrder,
    SalesOrderItem,
    Material,
    day_offset,
    SoldToParty,
    DistributionChannel,
    anomaly_code,
    'TAN' AS SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    BillingStatus,
    
    -- El producto 90025 ya consolidó su cambio histórico a L009 en el lote previo
    CASE 
        WHEN Material = '90025' THEN 'L009'
        ELSE concat('L00', cast(1 + pmod(abs(hash(Material, 'mgroup')), 3) AS STRING))
    END AS MaterialGroup,
    
    -- Disparador SCD Tipo 2 adicional: Material 90030 cambia a grupo C3 a partir de 2017 (día >= 731)
    CASE 
        WHEN Material = '90030' AND day_offset >= 731 THEN 'C3'
        WHEN pmod(abs(hash(Material, 'prgrp')), 2) = 0 THEN 'A1' 
        ELSE 'B2' 
    END AS MaterialPricingGroup,
    
    Material AS OriginallyRequestedMaterial,
    concat('CUST-MAT-', Material) AS MaterialByCustomer,
    
    cast(RequestedQuantity AS STRING) AS RequestedQuantity,
    cast(RequestedQuantity AS STRING) AS ConfdDelivQtyInOrderQtyUnit,
    cast(round(RequestedQuantity * UnitPrice, 2) AS STRING) AS NetAmount,
    cast(round((RequestedQuantity * UnitPrice) * 0.21, 2) AS STRING) AS TaxAmount,
    cast(round((RequestedQuantity * UnitPrice) * (1.0 - MarginPct), 2) AS STRING) AS CostAmount,
    cast(round(abs(RequestedQuantity) * 1.550, 3) AS STRING) AS ItemGrossWeight,
    cast(round(abs(RequestedQuantity) * 1.400, 3) AS STRING) AS ItemNetWeight,
    cast(round(abs(RequestedQuantity) * 0.035, 3) AS STRING) AS ItemVolume,
    
    -- Anomalía 3 (Radio 6 - Hard): Corrupción técnica
    CASE 
        WHEN anomaly_code = 2 AND SalesOrderItem = '10' THEN '{"error": "CORRUPTED_SYNTHETIC_PAYLOAD"}'
        ELSE NULL 
    END AS _rescued_data
FROM items_expanded;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW raw_synthetic_headers_prep AS
WITH header_aggregation AS (
    SELECT 
        order_id,
        SalesOrder,
        day_offset,
        SoldToParty,
        DistributionChannel,
        max(anomaly_code) AS anomaly_code,
        sum(cast(NetAmount AS DECIMAL(18, 2))) AS CalculatedTotalNet,
        max(_rescued_data) AS HeaderRescuedData
    FROM raw_synthetic_items_prep
    GROUP BY order_id, SalesOrder, day_offset, SoldToParty, DistributionChannel
),
dates_and_metadata AS (
    SELECT 
        SalesOrder,
        '9000' AS SalesOrganization,
        DistributionChannel,
        '09' AS OrganizationDivision,
        SoldToParty,
        
        -- Cliente 900015 continúa con sus términos renegociados (0001)
        -- Disparador SCD Tipo 2 adicional: Cliente 900040 cambia a términos 0002 en 2018 (día >= 1096)
        CASE 
            WHEN SoldToParty = '900015' THEN '0001'
            WHEN SoldToParty = '900040' AND day_offset >= 1096 THEN '0002'
            ELSE concat('000', cast(1 + pmod(abs(hash(SoldToParty, 'terms')), 4) AS STRING))
        END AS CustomerPaymentTerms,
        
        concat('0', cast(1 + pmod(abs(hash(SoldToParty, 'grp')), 4) AS STRING)) AS CustomerGroup,
        CASE WHEN DistributionChannel = '91' THEN '02' ELSE '01' END AS CustomerAccountAssignmentGroup,
        
        -- Simulación Soft Delete en origen (~2% cancelados 'C', resto 'A')
        CASE 
            WHEN pmod(abs(hash(order_id, 'rejection_seed')), 50) = 0 THEN 'C'
            ELSE 'A' 
        END AS OverallSDDocumentRejectionSts,
        
        -- Base 2015-01-01 08:00:00 UTC (1420102800000 ms) + offset calculado
        (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) + (pmod(abs(hash(order_id, 'sec')), 36000) * 1000)) AS CreationEpochMs,
        
        -- Anomalía 4 (Radio 6 - Hard): Entrega en el pasado
        CASE 
            WHEN anomaly_code = 3 THEN 
                (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) - (10 * 86400000))
            ELSE 
                (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) + (cast(3 + pmod(abs(hash(order_id, 'deliv_days')), 12) AS BIGINT) * 86400000))
        END AS DeliveryEpochMs,
        
        -- Anomalía 5 (Radio 5 - Soft): Descuadre contable
        CASE 
            WHEN anomaly_code = 4 THEN CalculatedTotalNet + 250.00
            ELSE CalculatedTotalNet 
        END AS TotalNetAmount,
        
        -- Anomalía 6 (Radio 3 - Hard): Moneda inválida
        CASE 
            WHEN anomaly_code = 5 THEN 'US_DOLLAR'
            ELSE 'USD'
        END AS TransactionCurrency,
        
        HeaderRescuedData AS _rescued_data
    FROM header_aggregation
)
SELECT 
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    OverallSDDocumentRejectionSts,
    concat('/Date(', cast(CreationEpochMs AS STRING), ')/') AS CreationDate,
    concat('/Date(', cast(CreationEpochMs AS STRING), ')/') AS SalesOrderDate,
    concat('/Date(', cast(DeliveryEpochMs AS STRING), ')/') AS RequestedDeliveryDate,
    cast(round(TotalNetAmount, 2) AS STRING) AS TotalNetAmount,
    TransactionCurrency,
    _rescued_data
FROM dates_and_metadata;

In [0]:
%sql
INSERT INTO workspace.bronze.sales_order_header (
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    OverallSDDocumentRejectionSts,
    CreationDate,
    SalesOrderDate,
    RequestedDeliveryDate,
    TotalNetAmount,
    TransactionCurrency,
    _rescued_data,
    _ingestion_timestamp,
    SourceSystem,
    IsSynthetic
)
SELECT 
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    OverallSDDocumentRejectionSts,
    CreationDate,
    SalesOrderDate,
    RequestedDeliveryDate,
    TotalNetAmount,
    TransactionCurrency,
    _rescued_data,
    current_timestamp() AS _ingestion_timestamp,
    'SYNTHETIC_GENERATOR' AS SourceSystem,
    TRUE AS IsSynthetic
FROM raw_synthetic_headers_prep;

In [0]:
%sql
INSERT INTO workspace.bronze.sales_order_item (
    SalesOrder,
    SalesOrderItem,
    Material,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    MaterialByCustomer,
    SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    OrderRelatedBillingStatus,
    RequestedQuantity,
    ConfdDelivQtyInOrderQtyUnit,
    NetAmount,
    TaxAmount,
    CostAmount,
    ItemGrossWeight,
    ItemNetWeight,
    ItemVolume,
    _rescued_data,
    _ingestion_timestamp,
    SourceSystem,
    IsSynthetic
)
-- Posiciones regulares
SELECT 
    SalesOrder,
    SalesOrderItem,
    Material,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    MaterialByCustomer,
    SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    BillingStatus,
    RequestedQuantity,
    ConfdDelivQtyInOrderQtyUnit,
    NetAmount,
    TaxAmount,
    CostAmount,
    ItemGrossWeight,
    ItemNetWeight,
    ItemVolume,
    _rescued_data,
    current_timestamp() AS _ingestion_timestamp,
    'SYNTHETIC_GENERATOR' AS SourceSystem,
    TRUE AS IsSynthetic
FROM raw_synthetic_items_prep

UNION ALL

-- Inyección de duplicados de prueba dentro del nuevo rango
SELECT 
    SalesOrder,
    SalesOrderItem,
    Material,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    MaterialByCustomer,
    SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    BillingStatus,
    RequestedQuantity,
    ConfdDelivQtyInOrderQtyUnit,
    NetAmount,
    TaxAmount,
    CostAmount,
    ItemGrossWeight,
    ItemNetWeight,
    ItemVolume,
    _rescued_data,
    current_timestamp() + INTERVAL 5 SECOND AS _ingestion_timestamp,
    'SYNTHETIC_GENERATOR' AS SourceSystem,
    TRUE AS IsSynthetic
FROM raw_synthetic_items_prep
WHERE SalesOrder IN ('9000100010', '9000100020', '9000100030')
  AND SalesOrderItem = '10';

Celda 5: Verificación de Continuidad Temporal
Ejecuta esta consulta tras la inserción para constatar que el rango de fechas en Bronze empalmó exactamente desde 2015 hasta finales de 2018:

In [0]:
%sql
SELECT 
    SourceSystem,
    IsSynthetic,
    COUNT(DISTINCT SalesOrder) AS TotalOrdenes,
    min(SalesOrder) AS MinOrden,
    max(SalesOrder) AS MaxOrden,
    -- Decodificación de fechas SAP en milisegundos a fecha legible
    to_date(from_unixtime(cast(regexp_extract(min(CreationDate), '([0-9]+)', 1) AS BIGINT) / 1000)) AS FechaMinima,
    to_date(from_unixtime(cast(regexp_extract(max(CreationDate), '([0-9]+)', 1) AS BIGINT) / 1000)) AS FechaMaxima
FROM workspace.bronze.sales_order_header
GROUP BY SourceSystem, IsSynthetic;

El resultado mostrará las 250.000 órdenes sintéticas totales cubriendo el periodo continuo desde el 01/01/2015 hasta el 31/12/2018. Al ejecutar el Workflow, el MERGE de Silver procesará el nuevo incremento de forma transparente.

Nuevo generador con correcciones de errores y modificaciones por escalabilidad y rendimiento.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW raw_synthetic_items_prep AS
WITH orders_seed AS (
    SELECT 
        id AS order_id,
        -- Numeración consecutiva: 9000000001 en adelante
        concat('90', lpad(cast(id AS STRING), 8, '0')) AS SalesOrder,
        -- Ventana continua de 4 años: 0 a 1460 días (01/01/2015 al 31/12/2018)
        pmod(abs(hash(id, 'day_seed')), 1461) AS day_offset,
        -- Cartera de 50 clientes sintéticos
        concat('900', lpad(cast(1 + pmod(abs(hash(id, 'cust_seed')), 50) AS STRING), 3, '0')) AS SoldToParty,
        -- Canales de distribución
        CASE WHEN pmod(abs(hash(id, 'dist_seed')), 10) < 7 THEN '90' ELSE '91' END AS DistributionChannel,
        -- 1 a 4 posiciones por pedido
        1 + pmod(abs(hash(id, 'items_cnt')), 4) AS num_items,
        pmod(abs(hash(id, 'anomaly_seed')), 200) AS anomaly_code
    FROM (SELECT explode(sequence(1, 250000)) AS id)
),
items_expanded AS (
    SELECT 
        o.order_id,
        o.SalesOrder,
        o.day_offset,
        o.SoldToParty,
        o.DistributionChannel,
        o.anomaly_code,
        item_idx,
        
        -- Anomalía 1 (Radio 2 - Hard): SalesOrderItem NULL
        CASE 
            WHEN o.anomaly_code = 0 AND item_idx = 1 THEN NULL 
            ELSE cast(item_idx * 10 AS STRING) 
        END AS SalesOrderItem,
        
        concat('900', lpad(cast(1 + pmod(abs(hash(o.order_id, item_idx, 'mat')), 40) AS STRING), 2, '0')) AS Material,
        
        -- Anomalía 2 (Radio 4 - Hard): Cantidad negativa
        CASE 
            WHEN o.anomaly_code = 1 AND item_idx = 1 THEN -5.0
            ELSE cast(1 + pmod(abs(hash(o.order_id, item_idx, 'qty')), 15) AS DECIMAL(18, 3))
        END AS RequestedQuantity,
        
        cast(40.0 + (pmod(abs(hash(o.order_id, item_idx, 'price')), 51000) / 100.0) AS DECIMAL(18, 2)) AS UnitPrice,
        cast(0.20 + (pmod(abs(hash(o.order_id, item_idx, 'margin')), 11) / 100.0) AS DECIMAL(18, 4)) AS MarginPct,
        
        CASE 
            WHEN pmod(abs(hash(o.order_id, item_idx, 'deliv_st')), 10) < 7 THEN 'C'
            WHEN pmod(abs(hash(o.order_id, item_idx, 'deliv_st')), 10) < 9 THEN 'B'
            ELSE 'A'
        END AS DeliveryStatus,
        
        CASE WHEN pmod(abs(hash(o.order_id, item_idx, 'proc_st')), 4) = 0 THEN 'B' ELSE 'C' END AS SDProcessStatus,
        CASE WHEN pmod(abs(hash(o.order_id, item_idx, 'bill_st')), 3) = 0 THEN 'A' ELSE 'C' END AS BillingStatus
    FROM orders_seed o
    LATERAL VIEW explode(sequence(1, o.num_items)) t AS item_idx
)
SELECT 
    order_id,
    SalesOrder,
    SalesOrderItem,
    Material,
    day_offset,
    SoldToParty,
    DistributionChannel,
    anomaly_code,
    'TAN' AS SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    BillingStatus,
    
    -- Disparador SCD Tipo 2 en Producto: Material 90025 cambia a L009 a partir del día 300
    CASE 
        WHEN Material = '90025' AND day_offset >= 300 THEN 'L009'
        ELSE concat('L00', cast(1 + pmod(abs(hash(Material, 'mgroup')), 3) AS STRING))
    END AS MaterialGroup,
    
    -- Disparador SCD Tipo 2 en Producto: Material 90030 cambia a C3 a partir del día 731 (año 2017)
    CASE 
        WHEN Material = '90030' AND day_offset >= 731 THEN 'C3'
        WHEN pmod(abs(hash(Material, 'prgrp')), 2) = 0 THEN 'A1' 
        ELSE 'B2' 
    END AS MaterialPricingGroup,
    
    Material AS OriginallyRequestedMaterial,
    concat('CUST-MAT-', Material) AS MaterialByCustomer,
    
    cast(RequestedQuantity AS STRING) AS RequestedQuantity,
    cast(RequestedQuantity AS STRING) AS ConfdDelivQtyInOrderQtyUnit,
    cast(round(RequestedQuantity * UnitPrice, 2) AS STRING) AS NetAmount,
    cast(round((RequestedQuantity * UnitPrice) * 0.21, 2) AS STRING) AS TaxAmount,
    cast(round((RequestedQuantity * UnitPrice) * (1.0 - MarginPct), 2) AS STRING) AS CostAmount,
    cast(round(abs(RequestedQuantity) * 1.550, 3) AS STRING) AS ItemGrossWeight,
    cast(round(abs(RequestedQuantity) * 1.400, 3) AS STRING) AS ItemNetWeight,
    cast(round(abs(RequestedQuantity) * 0.035, 3) AS STRING) AS ItemVolume,
    
    -- Anomalía 3 (Radio 6 - Hard): Corrupción técnica
    CASE 
        WHEN anomaly_code = 2 AND SalesOrderItem = '10' THEN '{"error": "CORRUPTED_SYNTHETIC_PAYLOAD"}'
        ELSE NULL 
    END AS _rescued_data
FROM items_expanded;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW raw_synthetic_headers_prep AS
WITH header_aggregation AS (
    SELECT 
        order_id,
        SalesOrder,
        day_offset,
        SoldToParty,
        DistributionChannel,
        max(anomaly_code) AS anomaly_code,
        sum(cast(NetAmount AS DECIMAL(18, 2))) AS CalculatedTotalNet,
        max(_rescued_data) AS HeaderRescuedData
    FROM raw_synthetic_items_prep
    GROUP BY order_id, SalesOrder, day_offset, SoldToParty, DistributionChannel
),
dates_and_metadata AS (
    SELECT 
        SalesOrder,
        '9000' AS SalesOrganization,
        DistributionChannel,
        '09' AS OrganizationDivision,
        SoldToParty,
        
        -- Disparadores SCD Tipo 2 en Clientes:
        -- Cliente 900015 cambia a '0001' a partir del día 250
        -- Cliente 900040 cambia a '0002' a partir del día 1096 (año 2018)
        CASE 
            WHEN SoldToParty = '900015' AND day_offset >= 250 THEN '0001'
            WHEN SoldToParty = '900015' THEN '0004'
            WHEN SoldToParty = '900040' AND day_offset >= 1096 THEN '0002'
            ELSE concat('000', cast(1 + pmod(abs(hash(SoldToParty, 'terms')), 4) AS STRING))
        END AS CustomerPaymentTerms,
        
        -- CustomerGroup fijo por cliente (01 a 04)
        concat('0', cast(1 + pmod(abs(hash(SoldToParty, 'grp')), 4) AS STRING)) AS CustomerGroup,
        
        -- Corrección DQ2: Atributo dependiente de SoldToParty.
        -- Conflicto intencional SOLO en 900049 y 900050 (~2% de clientes que violan DQ2)
        CASE 
            WHEN SoldToParty IN ('900049', '900050') AND DistributionChannel = '91' THEN '02'
            WHEN SoldToParty IN ('900049', '900050') THEN '01'
            WHEN pmod(abs(hash(SoldToParty, 'acct_grp')), 10) < 3 THEN '02'
            ELSE '01'
        END AS CustomerAccountAssignmentGroup,
        
        -- Soft Delete en origen (~2% cancelados 'C', resto 'A')
        CASE 
            WHEN pmod(abs(hash(order_id, 'rejection_seed')), 50) = 0 THEN 'C'
            ELSE 'A' 
        END AS OverallSDDocumentRejectionSts,
        
        -- Base 2015-01-01 08:00:00 UTC (1420102800000 ms)
        (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) + (pmod(abs(hash(order_id, 'sec')), 36000) * 1000)) AS CreationEpochMs,
        
        -- Anomalía 4 (Radio 6 - Hard): Entrega anterior a la creación
        CASE 
            WHEN anomaly_code = 3 THEN 
                (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) - (10 * 86400000))
            ELSE 
                (1420102800000 + (cast(day_offset AS BIGINT) * 86400000) + (cast(3 + pmod(abs(hash(order_id, 'deliv_days')), 12) AS BIGINT) * 86400000))
        END AS DeliveryEpochMs,
        
        -- Anomalía 5 (Radio 5 - Soft): Descuadre contable
        CASE 
            WHEN anomaly_code = 4 THEN CalculatedTotalNet + 250.00
            ELSE CalculatedTotalNet 
        END AS TotalNetAmount,
        
        -- Anomalía 6 (Radio 3 - Hard): Moneda inválida
        CASE 
            WHEN anomaly_code = 5 THEN 'US_DOLLAR'
            ELSE 'USD'
        END AS TransactionCurrency,
        
        HeaderRescuedData AS _rescued_data
    FROM header_aggregation
)
SELECT 
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    OverallSDDocumentRejectionSts,
    concat('/Date(', cast(CreationEpochMs AS STRING), ')/') AS CreationDate,
    concat('/Date(', cast(CreationEpochMs AS STRING), ')/') AS SalesOrderDate,
    concat('/Date(', cast(DeliveryEpochMs AS STRING), ')/') AS RequestedDeliveryDate,
    cast(round(TotalNetAmount, 2) AS STRING) AS TotalNetAmount,
    TransactionCurrency,
    _rescued_data
FROM dates_and_metadata;

In [0]:
%sql
INSERT INTO workspace.bronze.sales_order_header (
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    OverallSDDocumentRejectionSts,
    CreationDate,
    SalesOrderDate,
    RequestedDeliveryDate,
    TotalNetAmount,
    TransactionCurrency,
    _rescued_data,
    _ingestion_timestamp,
    SourceSystem,
    IsSynthetic
)
SELECT 
    SalesOrder,
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    SoldToParty,
    CustomerPaymentTerms,
    CustomerGroup,
    CustomerAccountAssignmentGroup,
    OverallSDDocumentRejectionSts,
    CreationDate,
    SalesOrderDate,
    RequestedDeliveryDate,
    TotalNetAmount,
    TransactionCurrency,
    _rescued_data,
    current_timestamp() AS _ingestion_timestamp,
    'SYNTHETIC_GENERATOR' AS SourceSystem,
    TRUE AS IsSynthetic
FROM raw_synthetic_headers_prep;

In [0]:
%sql
INSERT INTO workspace.bronze.sales_order_item (
    SalesOrder,
    SalesOrderItem,
    Material,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    MaterialByCustomer,
    SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    OrderRelatedBillingStatus,
    RequestedQuantity,
    ConfdDelivQtyInOrderQtyUnit,
    NetAmount,
    TaxAmount,
    CostAmount,
    ItemGrossWeight,
    ItemNetWeight,
    ItemVolume,
    _rescued_data,
    _ingestion_timestamp,
    SourceSystem,
    IsSynthetic
)
SELECT 
    SalesOrder,
    SalesOrderItem,
    Material,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    MaterialByCustomer,
    SalesOrderItemCategory,
    DeliveryStatus,
    SDProcessStatus,
    BillingStatus,
    RequestedQuantity,
    ConfdDelivQtyInOrderQtyUnit,
    NetAmount,
    TaxAmount,
    CostAmount,
    ItemGrossWeight,
    ItemNetWeight,
    ItemVolume,
    _rescued_data,
    current_timestamp() AS _ingestion_timestamp,
    'SYNTHETIC_GENERATOR' AS SourceSystem,
    TRUE AS IsSynthetic
FROM raw_synthetic_items_prep;